In [ ]:
import numpy as np
import cv2
from PIL import Image
from tqdm import tqdm
import os
import h5py


def normalize8(I):
    mn = I.min()
    mx = I.max()
    mx -= mn
    I = ((I - mn) / mx) * 255.0
    return np.round(I).astype(np.uint8)


def create_png(savedir, img_arrs, filename):
    # create '3-channeled' png file from image array
    max_val = []
    savedir = savedir
    n = len(img_arrs)
    for i in tqdm(range(n)):
        img_arr = img_arrs[i]
        max = img_arr.max()
        max_val.append(max)
        normalised = img_arr / max * 255.0
        arr_8bit = np.round(normalised).astype(np.uint8)
        stacked_img = np.stack((arr_8bit,) * 3, axis=-1)
        img = Image.fromarray(stacked_img)
        img.save(savedir + filename + str(i) + '.png')


def downsample(filelist, directory, savedir):
    # bicubic downsample the images 4 times on both axes
    filelist = os.listdir(filelist)
    directory = directory
    savedir = savedir
    for i in tqdm(filelist):
        img = cv2.imread(directory + i)
        img_resized = cv2.resize(img, None, fx=0.25, fy=0.25, interpolation=cv2.INTER_CUBIC)
        cv2.imwrite(savedir + i, img_resized)


def extract_h5(path):
    # extract and reshape the flattened array into image matrices
    f1 = h5py.File(path, 'r')
    # 3D MS image data
    liver1 = f1.get('counts_mat')
    liver1 = np.array(liver1)
    z = liver1.shape[1]
    dimxy = f1.get('dim_xy')
    dimxy = np.array(dimxy)
    image_arr = np.reshape(liver1, (dimxy[1], dimxy[0], z))
    return image_arr

In [ ]:
import h5py
import numpy as np


def noise_adding(path, savedir):
    # add artificial noise to image
    p = 0.5
    level = 50
    img_arr = np.load(path)

    lr0 = np.random.poisson(img_arr)
    gr0 = level * np.random.normal(0, 1, img_arr.shape) + img_arr
    nr0 = lr0 * p + gr0 * (1 - p)
    nr0 = np.clip(nr0, 0.0, 255.0)
    np.save(savedir, nr0)


def img2gray(arr):
    # intensity to 0-255
    Imax = np.max(arr)
    Imin = np.min(arr)
    k = 255.0 / (Imax - Imin)
    arr = k * (arr - Imin)
    return np.round(arr, 2)


def preselect(path):
    f1 = h5py.File(path, 'r')

    # 3D MS image data
    liver1 = f1.get('counts_mat')
    liver1 = np.array(liver1)
    z = liver1.shape[1]
    dimxy = f1.get('dim_xy')
    dimxy = np.array(dimxy)
    liver1 = np.reshape(liver1, (dimxy[1], dimxy[0], z))

    # Get the background noise
    hsnr = []
    lsnr = []
    snrlist = []
    mean_norm = []
    bg_list = []
    for i in range(z):
        gray = img2gray(liver1[:, :, i])
        mean = np.mean(gray)
        bg_list.append(mean)
    mi = np.argmin(bg_list)  # bg = the lowest mean intensity under gray image
    bg_gray = img2gray(liver1[:, :, mi])
    bg = np.mean(bg_gray)

    # Calculate SNR and filter out super low snr
    for i in range(z):
        gray = img2gray(liver1[:, :, i])
        mean = np.mean(gray)
        snr = mean / bg
        snr = 20 * np.log10(snr)
        snrlist.append(snr)
        mean_norm.append(mean)

    snrlist = np.array(snrlist)
    snr_mean = np.mean(snrlist)
    snr_med = np.median(snrlist)
    snr_standard = snr_mean

    for i in range(z):
        if snrlist[i] > snr_standard * 1:
            hsnr.append(i)
        else:
            lsnr.append(i)
    # Save the normalized gray image data
    for i in range(liver1.shape[2]):
        liver1[:, :, i] = img2gray(liver1[:, :, i])

    hr = np.delete(liver1, lsnr, axis=2)

    return hr